In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm


def read_binary_tif(path, threshold=0):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    assert img is not None, f"Failed to read {path}"
    return (img > threshold).astype(np.uint8)


def compute_confusion(pred, gt):

    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    TN = np.sum((pred == 0) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))
    return TP, FP, TN, FN


def compute_metrics(TP, FP, TN, FN, eps=1e-8):
    acc = (TP + TN) / (TP + FP + TN + FN + eps)
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = TP / (TP + FP + FN + eps)

    # Kappa
    total = TP + FP + TN + FN
    po = acc
    p1 = (TP + FN)*(TP + FP)
    p2 = (FP + TN)*(TN + FN)
    pe = (p1 + p2) / (total*total + eps)
    kappa = (po - pe) / (1 - pe + eps)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "IoU": iou,
        "Kappa": kappa
    }


def evaluate_one_model(pred_dir, gt_dir):
    TP = FP = TN = FN = 0

    names = sorted(os.listdir(gt_dir))

    # 遍历影像
    for name in tqdm(names):
        if not name.endswith(".tif"):
            continue

        gt_path = os.path.join(gt_dir, name)
        pred_path = os.path.join(pred_dir, name)

        if not os.path.exists(pred_path):
            print(f"[Warning] Missing pred: {name}")
            continue

        gt = read_binary_tif(gt_path)
        pred = read_binary_tif(pred_path)

        tTP, tFP, tTN, tFN = compute_confusion(pred, gt)
        TP += tTP
        FP += tFP
        TN += tTN
        FN += tFN

    return compute_metrics(TP, FP, TN, FN)



2022 dataset

In [ ]:

id = "20260123_114157_220_empty"  
comment = "### Learned query "


base_dir = "/work_dirs/output"
gt_dir = os.path.join(base_dir, "label")
result_csv = os.path.join(base_dir, "evaluation_results.csv")
                           
pred_dir = os.path.join(base_dir, id, "semantic")
metric = evaluate_one_model(pred_dir, gt_dir)



if not os.path.exists(result_csv):
    with open(result_csv, "w") as f:
        f.write("Model,Accuracy,Precision,Recall,F1,IoU,Kappa,Comment\n")
# 写入csv
with open(result_csv, "a") as f:
    f.write(
        f"{id},"
        f"{metric['Accuracy']:.6f},"
        f"{metric['Precision']:.6f},"
        f"{metric['Recall']:.6f},"
        f"{metric['F1']:.6f},"
        f"{metric['IoU']:.6f},"
        f"{metric['Kappa']:.6f},"
        f"\"{comment}\"\n")
print(f"PA: {metric['Precision']:.6f} | F1: {metric['F1']:.6f} | IoU: {metric['IoU']:.6f} | Kappa: {metric['Kappa']:.6f}")


0519 dataset

In [ ]:
# need update
id = "20260125_125848_40_target_empty"  
comment = "### 域自适应 "


base_dir = "/work_dirs/output"
gt_dir = os.path.join(base_dir, "label_0519")
result_csv = os.path.join(base_dir, "evaluation_results.csv")
                           
pred_dir = os.path.join(base_dir, id, "semantic")
metric = evaluate_one_model(pred_dir, gt_dir)


if not os.path.exists(result_csv):
    with open(result_csv, "w") as f:
        f.write("Model,Accuracy,Precision,Recall,F1,IoU,Kappa,Comment\n")

with open(result_csv, "a") as f:
    f.write(
        f"{id},"
        f"{metric['Accuracy']:.6f},"
        f"{metric['Precision']:.6f},"
        f"{metric['Recall']:.6f},"
        f"{metric['F1']:.6f},"
        f"{metric['IoU']:.6f},"
        f"{metric['Kappa']:.6f},"
        f"\"{comment}\"\n")
print(f"PA: {metric['Precision']:.6f} | F1: {metric['F1']:.6f} | IoU: {metric['IoU']:.6f} | Kappa: {metric['Kappa']:.6f}")
